# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging

logging.basicConfig(level=logging.INFO)

In [62]:
import gc
import logging
import os
from collections.abc import Callable
from copy import deepcopy
from dataclasses import dataclass
from functools import update_wrapper, wraps
from inspect import Parameter, signature
from time import perf_counter_ns
from types import MethodType
from typing import Any, Final, Optional, Union, overload, TypedDict
from typing import Dict, Any

import numpy as np

np.set_printoptions(precision=4, floatmode="fixed", suppress=True)
rng = np.random.default_rng()

Generator(PCG64) at 0x7FCE59CD6E40

In [3]:
import logging
from collections import OrderedDict
from typing import Any, TypeVar

import torch
from torch import Tensor, jit, nn

# from tsdm.models.generic.dense import ReverseDense
# from tsdm.util import deep_dict_update, initialize_from_config
# from tsdm.util.decorators import trace

INFO:numexpr.utils:Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


In [88]:
import dataclasses
from dataclasses import dataclass, field

import pydantic
from pydantic import BaseModel
from pydantic import dataclasses as pydantic_dataclasses
from pydantic.dataclasses import dataclass as pydantic_dataclass

# from dataclasses import KW_ONLY

In [14]:
@dataclass
class Config:
    input_size: int
    output_size: int
    latent_size: Optional[int] = None
    num_layers: int = 2
    activation: str = "relu"

In [16]:
dataclasses.asdict(Config(2,3))

{'input_size': 2,
 'output_size': 3,
 'latent_size': None,
 'num_layers': 2,
 'activation': 'relu'}

In [17]:
@pydantic_dataclass
class Config:
    input_size: int
    output_size: int
    latent_size: Optional[int] = None
    num_layers: int = 2
    activation: str = "relu"

In [18]:
dataclasses.asdict(Config(2,3))

{'input_size': 2,
 'output_size': 3,
 'latent_size': None,
 'num_layers': 2,
 'activation': 'relu'}

## Vanilla DataClasses

In [85]:
class MLP(nn.Sequential):
    HP: Dict[str, Any]

    @dataclass
    class Config:
        input_size: int
        output_size: int
        latent_size: Optional[int] = None
        num_layers: int = 2
        activation: str = "relu"

    def __init__(self, *args, **kwargs):
        config = self.Config(*args, **kwargs)
        
        if config.latent_size is None:
            config.latent_size = (config.input_size + config.output_size) // 2
        
        self.HP = dataclasses.asdict(config)

        layers = [nn.Linear(config.input_size, config.latent_size)]

        for _ in range(config.num_layers):
            layers.append(nn.Linear(config.latent_size, config.latent_size))

        layers.append(nn.Linear(config.latent_size, config.output_size))

        super().__init__(*layers)

In [86]:
model = MLP(2,3)
x = torch.randn(7,2)
model(x)
scripted = jit.script(model)
scripted(x)
jit.save(scripted, "model")
model = jit.load("model")
model.HP

{'input_size': 2,
 'output_size': 3,
 'latent_size': 2,
 'num_layers': 2,
 'activation': 'relu'}

## Pydantic DataClasses

In [76]:
class MLP(nn.Sequential):
    HP: Dict[str, Any]

    @pydantic_dataclass
    class Config:
        input_size: int
        output_size: int
        latent_size: Optional[int] = None
        num_layers: int = 2
        activation: str = "relu"

    def __init__(self, *args, **kwargs):
        config = self.Config(*args, **kwargs)
        
        if config.latent_size is None:
            config.latent_size = (config.input_size + config.output_size) // 2
        
        self.HP = dataclasses.asdict(config)

        layers = [nn.Linear(config.input_size, config.latent_size)]

        for _ in range(config.num_layers):
            layers.append(nn.Linear(config.latent_size, config.latent_size))

        layers.append(nn.Linear(config.latent_size, config.output_size))

        super().__init__(*layers)

In [77]:
model = MLP(2,3)
x = torch.randn(7,2)
model(x)
scripted = jit.script(model)
scripted(x)
jit.save(scripted, "model")
model = jit.load("model")
model.HP

{'input_size': 2,
 'output_size': 3,
 'latent_size': 2,
 'num_layers': 2,
 'activation': 'relu'}

## Pydantic BaseModel

In [78]:
class MLP(nn.Sequential):
    HP: Dict[str, Any]

    class Config(BaseModel):
        input_size: int
        output_size: int
        latent_size: Optional[int] = None
        num_layers: int = 2
        activation: str = "relu"

    def __init__(self, *args, **kwargs):
        config = self.Config(*args, **kwargs)
        
        if config.latent_size is None:
            config.latent_size = (config.input_size + config.output_size) // 2
        
        self.HP = dataclasses.asdict(config)

        layers = [nn.Linear(config.input_size, config.latent_size)]

        for _ in range(config.num_layers):
            layers.append(nn.Linear(config.latent_size, config.latent_size))

        layers.append(nn.Linear(config.latent_size, config.output_size))

        super().__init__(*layers)

# Nested Usage 

In [110]:
dataclasses.MISSING

In [120]:
from typing import TypeVar

In [119]:
@dataclass
class Config:
    input_size: int
    output_size: int
    latent_size: int
    
    def __post_init__(self):
        if self.latent_size is Any:
            self.latent_size = self.input_size

TypeError: typing.Any is not subscriptable

In [118]:
Config(2,3)

Config(input_size=2, output_size=3, latent_size=2)

In [104]:
@dataclass
class Config:
    input_size: int
    output_size: int
    latent_size: int = Any

    def __post_init__(self):
        if self.latent_size is Any:
            self.latent_size = self.input_size

conf = Config(2,3, latent_size=4)

Config(input_size=2, output_size=3, latent_size=4)

In [101]:
Config(2,3)

TypeError: __init__() missing 1 required positional argument: 'latent_size'

In [80]:
class Deepset(nn.Sequential):
    HP: Dict[str, Any]

    @dataclass
    class Config:
        input_size: int
        output_size: int
        latent_size: Optional[int] = None
        encoder:
        decoder:
        
    
    def __init__(self, *args, **kwargs) -> None:
        config = self.Config(*args, **kwargs)
        
        if config.latent_size is None:
            config.latent_size = (config.input_size + config.output_size) // 2

        self.HP = dataclasses.asdict(config)

    
    
    

SyntaxError: invalid syntax (1041022046.py, line 1)